# Gobernanza de Metadatos Semánticos y Knowledge Graphs (Aplicado)

## Objetivos
- Implementar gobierno de metadatos semánticos para discovery y reutilización.
- Construir un mini grafo de conocimiento (datasets, columnas, conceptos, dominios).
- Ejecutar consultas SPARQL para descubrir datasets relevantes por concepto/KPI.
- Validar contratos/propiedades mínimas de assets con verificaciones simples.
- Conectar con catálogo y data contracts para interoperabilidad.

## 1. Conceptos y Estándares
- **Metadatos semánticos**: describen significado (no solo formato) usando vocabularios controlados.
- **Catálogo** (DataHub/Atlas/Collibra): sistema de registro de assets.
- **DCAT**: vocabulario W3C para describir datasets y distribución.
- **SKOS**: vocabulario para taxonomías (concepts, broader/narrower/related).
- **RDF/OWL**: modelo de grafo y lenguaje de ontologías.
- **SPARQL**: consultas sobre grafos RDF.

## 2. Modelo de Grafo para Discovery/Reutilización
Tipos de nodos y relaciones:
- `Dataset` → `hasTable` → `Table` → `hasColumn` → `Column`.
- `Column` → `mapsToConcept` → `Concept` (SKOS/OWL).
- `Dataset` → `belongsToDomain` → `Domain`.
- `Dataset` → `partOfProduct` → `DataProduct`.
- `KPI` → `usesConcept` → `Concept`.
Esto permite responder: ¿qué datasets soportan el KPI X?, ¿qué columnas contienen PII?, ¿qué alternativas existen para reutilizar?

In [ ]:
# Construcción de un grafo semántico mínimo con rdflib
from rdflib import Graph, Namespace, RDF, RDFS, Literal, XSD, URIRef
from rdflib.namespace import SKOS, DCTERMS

EX = Namespace('https://example.com/data/')
g = Graph()
g.bind('ex', EX)
g.bind('skos', SKOS)
g.bind('dct', DCTERMS)

# Clases
for cls in ['Dataset','Table','Column','Concept','Domain','DataProduct','KPI']:
    g.add((EX[cls], RDF.type, RDFS.Class))

# Propiedades
props = ['hasTable','hasColumn','mapsToConcept','belongsToDomain','partOfProduct','usesConcept']
for p in props:
    g.add((EX[p], RDF.type, RDF.Property))

# Conceptos (taxonomía parcial)
customer = EX['concept/customer']
pii_email = EX['concept/pii/email']
revenue = EX['concept/revenue']
for c,label in [(customer,'Customer'),(pii_email,'PII Email'),(revenue,'Revenue')]:
    g.add((c, RDF.type, SKOS.Concept))
    g.add((c, SKOS.prefLabel, Literal(label)))

# Dominios y productos
dom_sales = EX['domain/sales']
dom_marketing = EX['domain/marketing']
dp_customer360 = EX['product/customer360']
for d in [dom_sales, dom_marketing, dp_customer360]:
    g.add((d, RDF.type, EX.Domain if 'domain' in str(d) else EX.DataProduct))

# Datasets, tablas, columnas
ds_orders = EX['dataset/sales_orders_silver']
tb_orders = EX['table/orders']
col_cust_id = EX['column/orders.customer_id']
col_amount = EX['column/orders.amount']

ds_customers = EX['dataset/marketing_customers_silver']
tb_customers = EX['table/customers']
col_email = EX['column/customers.email']
col_segment = EX['column/customers.segment']

# Instancias y relaciones
for ds in [ds_orders, ds_customers]:
    g.add((ds, RDF.type, EX.Dataset))
g.add((ds_orders, DCTERMS.title, Literal('Sales Orders (Silver)')))
g.add((ds_customers, DCTERMS.title, Literal('Marketing Customers (Silver)')))

g.add((tb_orders, RDF.type, EX.Table)); g.add((tb_customers, RDF.type, EX.Table))
g.add((col_cust_id, RDF.type, EX.Column)); g.add((col_amount, RDF.type, EX.Column))
g.add((col_email, RDF.type, EX.Column)); g.add((col_segment, RDF.type, EX.Column))

g.add((ds_orders, EX.hasTable, tb_orders)); g.add((tb_orders, EX.hasColumn, col_cust_id)); g.add((tb_orders, EX.hasColumn, col_amount))
g.add((ds_customers, EX.hasTable, tb_customers)); g.add((tb_customers, EX.hasColumn, col_email)); g.add((tb_customers, EX.hasColumn, col_segment))

# Semántica
g.add((col_cust_id, EX.mapsToConcept, customer))
g.add((col_email, EX.mapsToConcept, pii_email))
g.add((col_amount, EX.mapsToConcept, revenue))

# Dominio y data product
g.add((ds_orders, EX.belongsToDomain, dom_sales))
g.add((ds_customers, EX.belongsToDomain, dom_marketing))
g.add((ds_customers, EX.partOfProduct, dp_customer360))

# KPI
kpi_arpu = EX['kpi/arpu']
g.add((kpi_arpu, RDF.type, EX.KPI))
g.add((kpi_arpu, DCTERMS.title, Literal('ARPU')))
g.add((kpi_arpu, EX.usesConcept, revenue))
g.add((kpi_arpu, EX.usesConcept, customer))

print('Grafo cargado con nodos y relaciones clave.')

## 3. Consultas SPARQL de Descubrimiento/Reutilización
1) Datasets que soportan el KPI ARPU (por conceptos usados)
2) Datasets con columnas marcadas como PII (ej: email)
3) Alternativas de reutilización por dominio

In [ ]:
# 1) Datasets que soportan el KPI ARPU
q1 = """
PREFIX ex: <https://example.com/data/>
PREFIX dct: <http://purl.org/dc/terms/>
SELECT DISTINCT ?ds ?title
WHERE {
  ex:kpi/arpu ex:usesConcept ?concept .
  ?col a ex:Column ; ex:mapsToConcept ?concept .
  ?tbl ex:hasColumn ?col .
  ?ds a ex:Dataset ; ex:hasTable ?tbl ; dct:title ?title .
}
"""
for row in g.query(q1):
    print('KPI ARPU soportado por dataset:', row.title)

# 2) Datasets con PII (email)
q2 = """
PREFIX ex: <https://example.com/data/>
SELECT DISTINCT ?ds
WHERE {
  ?col a ex:Column ; ex:mapsToConcept ex:concept/pii/email .
  ?tbl ex:hasColumn ?col .
  ?ds a ex:Dataset ; ex:hasTable ?tbl .
}
"""
print('Datasets con PII (email):')
for row in g.query(q2):
    print('-', row.ds)

# 3) Datasets por dominio (reutilización por ownership)
q3 = """
PREFIX ex: <https://example.com/data/>
SELECT ?domain (COUNT(DISTINCT ?ds) AS ?datasets)
WHERE { ?ds a ex:Dataset ; ex:belongsToDomain ?domain . }
GROUP BY ?domain
"""
print('Conteo de datasets por dominio:')
for row in g.query(q3):
    print('-', row.domain, '→', int(row.datasets))

## 4. Validación Simple de Metadatos Requeridos
Validamos que cada `Dataset` tenga título y dominio asignado (simulación de SHACL mínima).

In [ ]:
# Validación ad-hoc (SHACL simplificado)
def validar_datasets(graph):
    problemas = []
    for ds in graph.subjects(RDF.type, EX.Dataset):
        title = list(graph.objects(ds, DCTERMS.title))
        domain = list(graph.objects(ds, EX.belongsToDomain))
        if not title or not domain:
            problemas.append((ds, 'Faltan metadatos: ' + ('title ' if not title else '') + ('domain' if not domain else '')))
    return problemas

issues = validar_datasets(g)
print('Problemas encontrados:' if issues else 'Todos los datasets cumplen metadatos mínimos')
for i in issues:
    print('-', i)

## 5. Integración con Catálogo y Contratos
- Mapear IDs del catálogo a IRIs del grafo (enlace estable).
- Enriquecer assets con `mapsToConcept` para mejorar búsqueda semántica.
- Publicar facetas de búsqueda por `Concept` y `Domain`.
- Exponer API: `GET /search?concept=customer` → datasets candidatos.

## 6. Ejercicio
1) Agrega un nuevo KPI `Churn` y enlázalo a conceptos `Customer` y `Status`.
2) Crea un dataset nuevo con columna `status` mapeada a `Status`.
3) Escribe una consulta para listar datasets que soportan el KPI Churn.
4) Extiende la validación para exigir `partOfProduct` cuando `belongsToDomain` sea `marketing`.